# Merge, clean, and audit translated game descriptions

Run this notebook after the nine translated batch CSVs are in `Translation Done`, alongside `no_translation_needed.csv`. It merges the ten files in source-row order, cleans residual HTML/entities, emojis, and repairable mojibake, then applies the agreed language rule: drop a game when confidently detected non-English content exceeds 20% of its description's letters; otherwise remove the detected non-English sentence/segment and keep the English text. Rows dropped by this rule or for unrecoverable replacement characters are logged to `Description_Audit_Dropped_Games.csv`.

The final audit checks the cleaned, retained descriptions and writes `Games_Description_English_Only.csv` only if the blocking checks pass. The notebook does not print descriptions and has not been run as part of this edit.


In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from html import unescape

from bs4 import BeautifulSoup
from ftfy import fix_text

import pandas as pd
import regex as unicode_regex
from langdetect import detect_langs, DetectorFactory
from IPython.display import display

DetectorFactory.seed = 0
APP_ID_COLUMN = 'AppID'
NAME_COLUMN = 'Name'
DESCRIPTION_COLUMN = 'About the game'
NUM_BATCHES = 9
MIN_SEGMENT_LETTERS = 25
MIN_LANGUAGE_CONFIDENCE = 0.80
MAX_NON_ENGLISH_SHARE = 0.20

prep_candidates = [Path('.'), Path('Data Prep'), Path('../Data Prep')]
prep_dir = next((candidate for candidate in prep_candidates if (candidate / 'Translation Done').is_dir() and (candidate / 'games_semi_processed.csv').is_file()), None)
if prep_dir is None:
    raise FileNotFoundError('Could not locate Data Prep/Translation Done. Start Jupyter from the project root or Data Prep folder.')
translation_done_dir = prep_dir / 'Translation Done'
output_path = prep_dir / 'Games_Description_English_Only.csv'
drop_log_path = prep_dir / 'Description_Audit_Dropped_Games.csv'
print('Translation input folder:', translation_done_dir.resolve())
print('Final output:', output_path.resolve())

In [ ]:
# Load all translated batches plus the rows that never needed translation.
batch_paths = [translation_done_dir / f'translation_batch_{i:02d}_of_{NUM_BATCHES:02d}.csv' for i in range(1, NUM_BATCHES + 1)]
no_translation_path = translation_done_dir / 'no_translation_needed.csv'
missing_paths = [path for path in [*batch_paths, no_translation_path] if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Missing required translation output file(s): ' + ', '.join(path.name for path in missing_paths))

batch_frames = [pd.read_csv(path, encoding='utf-8-sig', low_memory=False, dtype={APP_ID_COLUMN: 'string'}) for path in batch_paths]
no_translation = pd.read_csv(no_translation_path, encoding='utf-8-sig', low_memory=False, dtype={APP_ID_COLUMN: 'string'})
all_frames = [*batch_frames, no_translation]
expected_columns = list(all_frames[0].columns)
for path, frame in zip([*batch_paths, no_translation_path], all_frames):
    if list(frame.columns) != expected_columns:
        raise ValueError(f'Column mismatch in {path.name}; each file must retain the same columns.')
required_columns = {APP_ID_COLUMN, NAME_COLUMN, DESCRIPTION_COLUMN, '__source_row'}
missing_columns = sorted(required_columns - set(expected_columns))
if missing_columns:
    raise KeyError(f'Required columns missing from translation files: {missing_columns}')

merged = pd.concat(all_frames, ignore_index=True)
if merged[APP_ID_COLUMN].isna().any():
    raise ValueError('Merged files contain missing AppIDs.')
duplicate_appids = merged.loc[merged[APP_ID_COLUMN].duplicated(keep=False), [APP_ID_COLUMN, NAME_COLUMN]]
if not duplicate_appids.empty:
    display(duplicate_appids.head(20))
    raise ValueError(f'Merged files contain {merged[APP_ID_COLUMN].duplicated().sum():,} duplicate AppIDs.')
if merged['__source_row'].isna().any() or merged['__source_row'].duplicated().any():
    raise ValueError('Source row markers are missing or duplicated; the input partition may be incomplete.')
merged['__source_row'] = pd.to_numeric(merged['__source_row'], errors='raise')
merged = merged.sort_values('__source_row', kind='stable').reset_index(drop=True)
print(f'Rows merged: {len(merged):,}')
print(f'Translated batch rows: {sum(len(frame) for frame in batch_frames):,}')
print(f'No-translation rows: {len(no_translation):,}')

In [ ]:
# Clean residual artifacts and apply the agreed 20% non-English rule before the strict audit.
HTML_TAG_RE = re.compile(r'</?\s*[A-Za-z][^>]*>')
HTML_ENTITY_RE = re.compile(r'&(?:[A-Za-z][A-Za-z0-9]+|#\d+|#x[0-9A-Fa-f]+);')
EMOJI_RE = unicode_regex.compile(r'(?:\p{Extended_Pictographic}(?:\p{Emoji_Modifier})?(?:\uFE0F|\uFE0E)?(?:\u200D\p{Extended_Pictographic}(?:\p{Emoji_Modifier})?(?:\uFE0F|\uFE0E)?)*)|(?:\p{Regional_Indicator}{2})|(?:[#*0-9]\uFE0F?\u20E3)')
MOJIBAKE_MARKERS = ('Ã', 'Â', 'â€', 'ðŸ', 'ï¿½')
MOJIBAKE_RE = re.compile('|'.join(re.escape(marker) for marker in MOJIBAKE_MARKERS))
CJK_RE = unicode_regex.compile(r'(?:\p{Script=Han}|\p{Script=Hiragana}|\p{Script=Katakana}|\p{Script=Hangul})')
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+|(?<=[。！？])|[\r\n]+')
NON_ENGLISH_MARKERS = set('le les des est avec pour dans une jeu du et la los las del para pero juego una con que nicht der die das und ist ein spiel mit il lo gli che gioco per nao voce het een van zijn voor niet'.split())


def clean_description(value):
    text = '' if pd.isna(value) else str(value)
    original = text
    for _ in range(2):
        decoded = unescape(text)
        if decoded == text:
            break
        text = decoded

    mojibake_candidate = bool(MOJIBAKE_RE.search(text))
    if mojibake_candidate:
        text = fix_text(text)

    html_stripped = bool(HTML_TAG_RE.search(text))
    if html_stripped:
        soup = BeautifulSoup(text, 'html.parser')
        for element in soup(['script', 'style']):
            element.decompose()
        text = soup.get_text(separator=' ', strip=True)

    # Strip any malformed or tag-like fragments the HTML parser preserved.
    residual_tag_fragments = len(HTML_TAG_RE.findall(text))
    if residual_tag_fragments:
        text = HTML_TAG_RE.sub(' ', text)

    emoji_count = len(EMOJI_RE.findall(text))
    if emoji_count:
        text = EMOJI_RE.sub(' ', text)

    replacement_count = text.count('\ufffd')
    remaining_mojibake = bool(MOJIBAKE_RE.search(text))
    if replacement_count:
        text = text.replace('\ufffd', ' ')

    output_chars = []
    for char in text:
        if unicodedata.category(char) == 'Cc' and char not in '\t\r\n':
            output_chars.append(' ')
        else:
            output_chars.append(char)
    text = ''.join(output_chars).replace('\r', ' ').replace('\n', ' ').replace('\t', ' ')
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return {
        'cleaned': text,
        'html_stripped': html_stripped,
        'residual_tag_fragments_removed': residual_tag_fragments,
        'emoji_count': emoji_count,
        'mojibake_candidate': mojibake_candidate,
        'remaining_mojibake': remaining_mojibake,
        'replacement_count': replacement_count,
        'text_changed': text != original,
    }


def likely_needs_language_detection(text):
    text = '' if pd.isna(text) else str(text)
    if CJK_RE.search(text):
        return True
    letters = [char for char in text if char.isalpha()]
    non_ascii_letters = sum(ord(char) > 127 for char in letters)
    if non_ascii_letters >= 4 and non_ascii_letters / max(1, len(letters)) >= 0.02:
        return True
    words = set(unicode_regex.findall(r'\p{L}+', text.lower()))
    return len(words & NON_ENGLISH_MARKERS) >= 2


def count_letters(text):
    return sum(char.isalpha() for char in text)


def analyze_language_segments(text):
    text = '' if pd.isna(text) else str(text)
    segments = [part.strip() for part in SENTENCE_SPLIT_RE.split(text) if part.strip()]
    total_letters = sum(count_letters(part) for part in segments)
    non_english_letters = 0
    non_english_segments = []
    for segment in segments:
        letters = count_letters(segment)
        if letters < MIN_SEGMENT_LETTERS:
            continue
        try:
            predictions = detect_langs(segment[:1500])
        except Exception:
            continue
        if predictions and predictions[0].prob >= MIN_LANGUAGE_CONFIDENCE and predictions[0].lang != 'en':
            non_english_letters += letters
            non_english_segments.append(segment)
    share = non_english_letters / total_letters if total_letters else 0.0
    return segments, non_english_segments, share


# First normalize markup/emoji/mojibake and keep a metadata record for the action log.
clean_records = []
for position, description in enumerate(merged[DESCRIPTION_COLUMN], start=1):
    clean_records.append(clean_description(description))
    if position % 10000 == 0:
        print(f'Code cleanup progress: {position:,}/{len(merged):,}')

clean_metadata = pd.DataFrame(clean_records, index=merged.index)
merged[DESCRIPTION_COLUMN] = clean_metadata['cleaned']

drop_reasons = {}
drop_details = {}
removed_non_english_rows = 0
removed_non_english_segments = 0
above_threshold_rows = 0
for position, text in enumerate(merged[DESCRIPTION_COLUMN].fillna('').astype(str)):
    app_id = merged.iloc[position][APP_ID_COLUMN]
    name = merged.iloc[position][NAME_COLUMN]
    reasons = []
    share = 0.0
    segments = []
    non_english_segments = []

    if bool(clean_metadata.iloc[position]['replacement_count']):
        reasons.append('unrecoverable replacement character')
    if bool(clean_metadata.iloc[position]['remaining_mojibake']):
        reasons.append('mojibake remained after repair attempt')
    if not text.strip():
        reasons.append('description empty after cleanup')

    if text.strip() and likely_needs_language_detection(text):
        segments, non_english_segments, share = analyze_language_segments(text)
        if share > MAX_NON_ENGLISH_SHARE:
            above_threshold_rows += 1
            reasons.append('non-English letter share above 20%')
        elif non_english_segments:
            # Remove confidently detected foreign-language sentence/segments while retaining the English parts.
            foreign_segments = set(non_english_segments)
            kept_segments = [segment for segment in segments if segment not in foreign_segments]
            updated_text = re.sub(r'\s+', ' ', ' '.join(kept_segments)).strip()
            merged.iat[position, merged.columns.get_loc(DESCRIPTION_COLUMN)] = updated_text
            removed_non_english_rows += 1
            removed_non_english_segments += len(non_english_segments)
            if not updated_text:
                reasons.append('description empty after removing non-English segments')

    if reasons:
        drop_reasons[position] = '; '.join(reasons)
        drop_details[position] = {
            APP_ID_COLUMN: app_id,
            NAME_COLUMN: name,
            'Non-English letter share': share,
            'Non-English segment count': len(non_english_segments),
            'Replacement characters after repair': int(clean_metadata.iloc[position]['replacement_count']),
            'Reason': drop_reasons[position],
        }

if drop_reasons:
    drop_log = pd.DataFrame.from_dict(drop_details, orient='index').reset_index(drop=True)
else:
    drop_log = pd.DataFrame(columns=[APP_ID_COLUMN, NAME_COLUMN, 'Non-English letter share', 'Non-English segment count', 'Replacement characters after repair', 'Reason'])
drop_log.to_csv(drop_log_path, index=False, encoding='utf-8-sig')
rows_dropped = len(drop_log)
if rows_dropped:
    merged = merged.drop(index=list(drop_reasons)).reset_index(drop=True)

print('Rows with HTML removed by parser:', int(clean_metadata['html_stripped'].sum()))
print('Residual tag-like fragments removed:', int(clean_metadata['residual_tag_fragments_removed'].sum()))
print('Emoji matches removed:', int(clean_metadata['emoji_count'].sum()))
print('Rows with mojibake repair attempted:', int(clean_metadata['mojibake_candidate'].sum()))
print('Rows dropped for unrecoverable replacement characters:', int(clean_metadata['replacement_count'].gt(0).sum()))
print(f'Rows over the 20% non-English threshold: {above_threshold_rows:,}')
print(f'Rows with non-English segments removed (at or below 20%): {removed_non_english_rows:,}')
print(f'Non-English segments removed: {removed_non_english_segments:,}')
print(f'Total rows dropped: {rows_dropped:,}')
print('Drop log:', drop_log_path.resolve())


In [ ]:
# Strictly audit the cleaned, retained descriptions before saving the final dataset.
texts = merged[DESCRIPTION_COLUMN].fillna('').astype(str)
residual_non_english = []
for position, text in enumerate(texts):
    if likely_needs_language_detection(text):
        _, segments, share = analyze_language_segments(text)
        if share > 0:
            residual_non_english.append((merged.iloc[position][APP_ID_COLUMN], merged.iloc[position][NAME_COLUMN], share, len(segments)))
    if (position + 1) % 5000 == 0:
        print(f'Final audit progress: {position + 1:,}/{len(merged):,}')

audit_counts = {
    'rows': len(merged),
    'empty_descriptions': int(texts.str.strip().eq('').sum()),
    'emoji_matches': int(texts.map(lambda text: len(EMOJI_RE.findall(text))).sum()),
    'rows_with_html_tags': int(texts.map(lambda text: bool(HTML_TAG_RE.search(text))).sum()),
    'rows_with_html_entities': int(texts.map(lambda text: bool(HTML_ENTITY_RE.search(text))).sum()),
    'rows_with_mojibake_markers': int(texts.map(lambda text: bool(MOJIBAKE_RE.search(text))).sum()),
    'rows_with_replacement_characters': int(texts.str.contains('\ufffd', regex=False).sum()),
    'rows_with_non_ascii_letters': int(texts.map(lambda text: any(ord(char) > 127 and char.isalpha() for char in text)).sum()),
    'rows_with_confident_non_english_segments': len(residual_non_english),
}
print('Final description audit:', json.dumps(audit_counts, ensure_ascii=False))
if residual_non_english:
    print('Residual language flags (AppID/name only; descriptions are omitted):')
    display(pd.DataFrame(residual_non_english[:50], columns=['AppID', 'Name', 'Non-English letter share', 'Non-English segment count']))

blocking_issues = {
    key: value for key, value in audit_counts.items()
    if key in {'empty_descriptions', 'emoji_matches', 'rows_with_html_tags', 'rows_with_html_entities', 'rows_with_mojibake_markers', 'rows_with_replacement_characters', 'rows_with_confident_non_english_segments'} and value
}
if blocking_issues:
    raise RuntimeError('Final audit found remaining description issues; review/fix them before saving: ' + json.dumps(blocking_issues, ensure_ascii=False))
print('Final description audit passed.')


In [ ]:
# Save only after the merged dataset passes the description audit.
final = merged.drop(columns=['__source_row']).copy()
temp_output = output_path.with_name(output_path.stem + '.tmp.csv')
try:
    final.to_csv(temp_output, index=False, encoding='utf-8-sig')
    temp_output.replace(output_path)
except Exception:
    if temp_output.exists():
        temp_output.unlink()
    raise
print(f'Saved audited English-only dataset: {len(final):,} games')
print('Output:', output_path.resolve())